# Notebook 2: Modeling dan Evaluasi Inventory Policy (RQ2)

**Pertanyaan yang dijawab (RQ2):** apakah model machine learning yang memperkirakan pemakaian bahan, lalu dimasukkan ke aturan inventory policy, menghasilkan status stok (Restock, Sufficient, Excess) yang lebih tepat dibanding baseline tanpa model?

**Input:** `dataset_model.csv` dan `config_simulasi.json` dari Notebook 1 (data prep).
**Output:** model final `usage_model_gb_q07.joblib` dan tabel `hasil_evaluasi_test.csv`.

### Ide utama: model memperkirakan ANGKA, label dibuat oleh aturan

Model di notebook ini adalah **model regresi**, bukan model klasifikasi. Model memperkirakan berapa banyak bahan yang akan terpakai dalam 7 hari ke depan. Angka itu kemudian dimasukkan ke inventory policy yang sama dengan yang dipakai untuk membuat label di Notebook 1.

```
fitur pagi hari  →  MODEL REGRESI  →  perkiraan pemakaian 7 hari (angka)
                                          ↓
                    inventory policy: stok gudang + sedang dikirim − perkiraan pemakaian
                                          ↓
                    sisa stok (dalam hari)  →  label Restock / Sufficient / Excess
```

Dengan cara ini, owner tidak hanya mendapat status, tetapi juga **angka kekurangan atau sisa** yang dipakai di laporan GenAI (Notebook 3).

### Aturan main evaluasi
1. Data dibagi **berdasarkan waktu**: train sebelum 1 Juni 2022, test mulai 1 Juni 2022. Model tidak pernah melihat masa depan.
2. Pemilihan model dilakukan di **data validasi** (bagian akhir data train).
3. Kandidat **dikunci dulu**, baru data test dibuka **satu kali** di Cell 7.
4. Metrik utama dilihat **per kelas** (recall dan F1 Restock, F1 Excess) dan **F1 macro**, bukan hanya akurasi, karena kelas Excess jumlahnya sedikit.

## Cell 1: Setup dan pengecekan dataset

**Tujuan:** membaca dataset dan parameter simulasi dari Notebook 1, lalu memastikan datanya sama dengan yang disimpan di akhir notebook tersebut.

**Yang perlu diperhatikan dari hasil:**
- Dataset berisi 6.286 baris (10 bahan × hari), dengan 5.126 baris train dan 1.160 baris test.
- Proporsi label tidak seimbang. Di train: Sufficient 66,4%, Restock 29,5%, **Excess hanya 4,1%**. Di test, Excess naik menjadi 7,8%.
- Kolom `split` sesuai dengan `SPLIT_DATE`, jadi pembagian train dan test benar berdasarkan tanggal.

**Kenapa tidak memakai oversampling atau undersampling?** Teknik seperti SMOTE dipakai untuk model klasifikasi. Model di sini adalah regresi pemakaian, sedangkan label baru muncul setelah policy diterapkan, sehingga tidak ada kelas yang bisa di-resample saat training. Ketidakseimbangan ditangani lewat **metrik per kelas dan F1 macro**, serta lewat **quantile** di Cell 5.

In [ ]:

# CELL 1 : Setup, baca dataset dan config dari notebook data prep

from google.colab import drive
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

# 1. Folder proyek
PROJECT_DIR   = Path("/content/drive/MyDrive/Tewiwi-Bakehouse")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR     = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)   # buat folder models kalau belum ada

# 2. Baca dataset hasil notebook data prep
data = pd.read_csv(PROCESSED_DIR / "dataset_model.csv", parse_dates=["date"])

# 3. Baca "kartu resep" (parameter simulasi dan aturan label)
with open(PROCESSED_DIR / "config_simulasi.json") as f:
    CFG = json.load(f)

FITUR        = CFG["FITUR"]
SPLIT_DATE   = pd.Timestamp(CFG["SPLIT_DATE"])
SEED         = CFG["SEED"]
H            = CFG["H"]
RESTOCK_DAYS = CFG["RESTOCK_DAYS"]
EXCESS_DAYS  = CFG["EXCESS_DAYS"]

print("==== Ukuran dataset ====")
print(data.shape)

print("\n==== Parameter dari notebook data prep ====")
for k, v in CFG.items():
    if k not in ("SAFETY_DAYS", "FITUR"):
        print(f"{k:13s}: {v}")
print("FITUR        :", FITUR)

# 4. Cek konsistensi: jumlah baris dan proporsi label harus sama dengan Cell 32
print("\n==== Jumlah baris per periode ====")
print(data["split"].value_counts())

print("\n==== Proporsi label per periode (%) ====")
display(pd.crosstab(data["split"], data["label"], normalize="index").mul(100).round(1))

# 5. Pastikan kolom split sesuai tanggal
cek_split = (data["split"] == np.where(data["date"] < SPLIT_DATE, "train", "test")).all()
print("\nKolom split sesuai SPLIT_DATE:", cek_split)

Mounted at /content/drive
==== Ukuran dataset ====
(6286, 15)

==== Parameter dari notebook data prep ====
SEED         : 42
H            : 7
SPLIT_DATE   : 2022-06-01
LEAD_TIME    : 3
ORDER_DAYS   : 7
WASTE_PCT    : 0.05
SERVICE_Z    : 1.65
BULK_Q       : 0.999
RESTOCK_DAYS : 0
EXCESS_DAYS  : 7
FITUR        : ['avg_usage_7', 'avg_usage_30', 'stok_now', 'on_order', 'days_since_last_order', 'stok_cover_now', 'posisi_stok']

==== Jumlah baris per periode ====
split
train    5126
test     1160
Name: count, dtype: int64

==== Proporsi label per periode (%) ====


label,Excess,Restock,Sufficient
split,,,
test,7.8,32.8,59.5
train,4.1,29.5,66.4



Kolom split sesuai SPLIT_DATE: True


## Cell 2: Baseline inventory policy (tanpa machine learning)

**Tujuan:** membuat pembanding yang adil. Baseline memakai cara paling sederhana yang biasa dipakai usaha kecil: **pemakaian 7 hari ke depan = rata-rata pemakaian 30 hari terakhir × 7**.

**Dua fungsi penting yang dipakai sampai Notebook 3:**
- `beri_label(sisa_days)`: sisa di bawah 0 hari → Restock, di atas 7 hari → Excess, di antaranya → Sufficient.
- `policy(df, pred_usage)`: menghitung sisa stok dari perkiraan pemakaian, lalu memberi label.

Baseline dan model memakai **policy yang sama persis**. Yang berbeda hanya perkiraan pemakaiannya, jadi perbandingannya adil.

**Pengecekan konsistensi:** policy + pemakaian aktual menghasilkan label asli (`True`). Artinya label sepenuhnya ditentukan oleh pemakaian, sehingga **kunci keberhasilan ada pada ketepatan perkiraan pemakaian**.

**Hasil baseline di test:**
- Rata-rata meleset 2,65 hari pemakaian.
- Recall Restock 0,516: hampir separuh kondisi yang seharusnya Restock tidak terdeteksi.
- F1 Excess 0: baseline hanya 3 kali menebak Excess dan ketiganya salah.
- F1 macro 0,434.

In [ ]:
# CELL 2 : Baseline inventory policy (tanpa model ML)

from sklearn.metrics import classification_report, confusion_matrix

LABELS = ["Restock", "Sufficient", "Excess"]

# 1. Aturan label (sama persis dengan notebook data prep)
def beri_label(sisa_days):
    return np.select(
        [sisa_days < RESTOCK_DAYS, sisa_days > EXCESS_DAYS],
        ["Restock", "Excess"],
        default="Sufficient",
    )

# 2. Inventory policy: tebakan pemakaian → proyeksi stok → sisa hari → label
def policy(df, pred_usage):
    proyeksi = df["stok_now"] + df["on_order"] - pred_usage
    sisa_days = proyeksi / df["avg_usage_30"]
    return sisa_days, beri_label(sisa_days)

# 3. Cek konsistensi: policy + pemakaian sebenarnya harus menghasilkan label asli
_, label_cek = policy(data, data["target_usage_7"])
print("Policy + pemakaian aktual = label asli:", (label_cek == data["label"]).all())

# 4. Baseline: tebakan pemakaian = rata-rata 30 hari x 7
data["pred_baseline"] = data["avg_usage_30"] * H
data["sisa_baseline"], data["label_baseline"] = policy(data, data["pred_baseline"])

# 5. Error tebakan pemakaian, dalam satuan "hari pemakaian" (bisa dibandingkan antar bahan)
data["err_baseline_hari"] = (data["pred_baseline"] - data["target_usage_7"]).abs() / data["avg_usage_30"]

test = data[data["split"] == "test"]

print("\n==== Error tebakan pemakaian baseline (test) ====")
print(f"Rata-rata meleset: {test['err_baseline_hari'].mean():.2f} hari pemakaian")

print("\n==== Evaluasi label baseline (test) ====")
print(classification_report(test["label"], test["label_baseline"], labels=LABELS, digits=3))

print("==== Confusion matrix baseline (test) ====")
cm = confusion_matrix(test["label"], test["label_baseline"], labels=LABELS)
display(pd.DataFrame(
    cm,
    index=[f"aktual {l}" for l in LABELS],
    columns=[f"tebakan {l}" for l in LABELS],
))

Policy + pemakaian aktual = label asli: True

==== Error tebakan pemakaian baseline (test) ====
Rata-rata meleset: 2.65 hari pemakaian

==== Evaluasi label baseline (test) ====
              precision    recall  f1-score   support

     Restock      0.616     0.516     0.562       380
  Sufficient      0.673     0.819     0.739       690
      Excess      0.000     0.000     0.000        90

    accuracy                          0.656      1160
   macro avg      0.430     0.445     0.434      1160
weighted avg      0.602     0.656     0.624      1160

==== Confusion matrix baseline (test) ====


,tebakan Restock,tebakan Sufficient,tebakan Excess
aktual Restock,196,184,0
aktual Sufficient,122,565,3
aktual Excess,0,90,0


## Cell 3: Fitur dan target dalam satuan "hari pemakaian"

**Masalah:** terigu dipakai ratusan kilogram per minggu, sedangkan ragi hanya beberapa kilogram. Kalau dipelajari dalam gram, model akan didominasi bahan bervolume besar.

**Solusi:** semua fitur dan target diubah ke satuan **hari pemakaian rata-rata** (dibagi `avg_usage_30`). Dengan begitu, 10 bahan bisa dipelajari oleh satu model yang sama.

| Fitur | Arti |
|---|---|
| `tren_7_vs_30` | rata-rata 7 hari dibanding 30 hari. Di atas 1 berarti minggu ini lebih ramai dari biasanya |
| `stok_cover_now` | stok gudang cukup untuk berapa hari |
| `on_order_hari` | bahan yang sedang dikirim setara berapa hari |
| `posisi_stok` | (stok gudang + sedang dikirim) dibanding titik pemesanan ulang (ROP) |
| `days_since_last_order` | sudah berapa hari sejak pemesanan terakhir |
| `bulan` | pola musiman. Informasi ini sudah diketahui di pagi hari |

**Target:** `target_rasio` = pemakaian 7 hari ke depan ÷ `avg_usage_30`. Baseline dalam satuan ini selalu bernilai 7.

**Pencegahan data leakage:** semua fitur hanya memakai informasi yang tersedia **pagi hari sebelum operasional** (dibuat dengan `shift(1)` di Notebook 1). Target memakai data masa depan, dan itu memang seharusnya untuk target.

**Pengecekan:** meleset baseline dalam satuan rasio = 2,65 hari, sama dengan Cell 2. Konversi satuan benar.

In [ ]:
# CELL 3 : Siapkan fitur dan target model (dalam satuan "hari pemakaian")

# 1. Fitur tanpa satuan gram, supaya terigu dan telur bisa dipelajari bersama
def buat_fitur_model(df):
    X = pd.DataFrame(index=df.index)
    X["tren_7_vs_30"]          = df["avg_usage_7"] / df["avg_usage_30"]   # > 1: seminggu ini lebih ramai dari sebulan
    X["stok_cover_now"]        = df["stok_cover_now"]                     # stok gudang cukup berapa hari
    X["on_order_hari"]         = df["on_order"] / df["avg_usage_30"]      # barang di jalan setara berapa hari
    X["posisi_stok"]           = df["posisi_stok"]                        # (gudang + di jalan) dibanding garis merah
    X["days_since_last_order"] = df["days_since_last_order"]              # sudah berapa hari sejak pesan
    X["bulan"]                 = df["date"].dt.month                      # pola musim (diketahui di pagi hari)
    return X

# 2. Target: pemakaian 7 hari ke depan, dalam satuan "hari pemakaian rata-rata"
data["target_rasio"] = data["target_usage_7"] / data["avg_usage_30"]

train = data[data["split"] == "train"].copy()
test  = data[data["split"] == "test"].copy()

X_train, y_train = buat_fitur_model(train), train["target_rasio"]
X_test,  y_test  = buat_fitur_model(test),  test["target_rasio"]
FITUR_MODEL = list(X_train.columns)

print("Ukuran X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Fitur model:", FITUR_MODEL)

print("\n==== Ringkasan fitur (train) ====")
display(X_train.describe().T[["mean", "min", "50%", "max"]].round(2))

print("\n==== Sebaran target_rasio ====")
print(pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()}).round(2))

# 3. Cek konsistensi: baseline dalam satuan rasio = selalu 7
#    Meleset rata-ratanya harus sama dengan Cell 2 (2,65 hari)
print(f"\nMeleset baseline (rasio): {(7 - y_test).abs().mean():.2f} hari")

Ukuran X_train: (5126, 6) | X_test: (1160, 6)
Fitur model: ['tren_7_vs_30', 'stok_cover_now', 'on_order_hari', 'posisi_stok', 'days_since_last_order', 'bulan']

==== Ringkasan fitur (train) ====


,mean,min,50%,max
tren_7_vs_30,1.02,0.52,0.99,2.54
stok_cover_now,5.72,0.00,5.56,17.46
on_order_hari,2.83,0.00,0.00,12.00
posisi_stok,1.79,0.93,1.79,3.46
days_since_last_order,4.96,1.00,4.00,22.00
bulan,5.53,1.00,5.00,12.00



==== Sebaran target_rasio ====
         train     test
count  5126.00  1160.00
mean      6.90     7.29
std       2.38     3.30
min       2.58     2.29
25%       5.20     4.73
50%       6.43     6.62
75%       8.11     9.41
max      17.84    19.61

Meleset baseline (rasio): 2.65 hari


## Cell 4: Membandingkan beberapa model di data validasi

**Tujuan:** memilih jenis model tanpa menyentuh data test.

**Pembagian berbasis waktu di dalam data train:**
```
train dalam (s/d Januari 2022, 3.926 baris)
validasi (Feb s/d Mei 2022, 1.200 baris)
test (Jun s/d Sep 2022)
```

**Fungsi `evaluasi`** menilai model dengan jalur yang sama seperti penggunaan nyata: perkiraan rasio → perkiraan pemakaian → policy → label → metrik.

**Hasil di validasi:**
- Gradient Boosting punya F1 macro tertinggi (0,571) dan satu-satunya yang bisa mendeteksi sebagian Excess (F1 0,267).
- Untuk Restock, baseline justru sedikit lebih baik di periode validasi (recall 0,661).
- Data validasi hanya punya **8 baris Excess**, sehingga F1 Excess di validasi tidak stabil dan tidak bisa dijadikan patokan tunggal.

In [ ]:
# CELL 4 : Latih beberapa model dan bandingkan di data validasi

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import f1_score, recall_score

# 1. Bagi train menjadi train dalam dan validasi (berbasis waktu)
VAL_DATE = pd.Timestamp("2022-02-01")
tr = train[train["date"] < VAL_DATE]
va = train[train["date"] >= VAL_DATE]

X_tr, y_tr = buat_fitur_model(tr), tr["target_rasio"]
X_va, y_va = buat_fitur_model(va), va["target_rasio"]
print(f"Train dalam: {len(tr)} baris | Validasi: {len(va)} baris")

# 2. Fungsi evaluasi: tebakan rasio → pemakaian → policy → label → metrik
def evaluasi(df, pred_rasio):
    pred_usage = pred_rasio * df["avg_usage_30"]
    _, label_pred = policy(df, pred_usage)
    y_true = df["label"]
    return {
        "meleset_hari":   np.abs(pred_rasio - df["target_rasio"]).mean(),
        "recall_restock": recall_score(y_true, label_pred, labels=["Restock"], average=None, zero_division=0)[0],
        "f1_restock":     f1_score(y_true, label_pred, labels=["Restock"], average=None, zero_division=0)[0],
        "f1_excess":      f1_score(y_true, label_pred, labels=["Excess"], average=None, zero_division=0)[0],
        "f1_macro":       f1_score(y_true, label_pred, labels=LABELS, average="macro", zero_division=0),
    }

# 3. Daftar model yang dicoba
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=SEED, n_jobs=-1
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=SEED),
}

# 4. Baseline di data validasi (selalu menebak 7)
hasil = {"Baseline (rata-rata x 7)": evaluasi(va, np.full(len(va), float(H)))}

# 5. Latih tiap model di train dalam, nilai di validasi
for nama, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    hasil[nama] = evaluasi(va, pred)

print("\n==== Perbandingan di data validasi ====")
display(pd.DataFrame(hasil).T.round(3))

Train dalam: 3926 baris | Validasi: 1200 baris

==== Perbandingan di data validasi ====


,meleset_hari,recall_restock,f1_restock,f1_excess,f1_macro
Baseline (rata-rata x 7),1.719,0.661,0.728,0.000,0.524
Linear Regression,1.643,0.629,0.711,0.000,0.517
Random Forest,1.909,0.591,0.645,0.000,0.479
Gradient Boosting,1.848,0.595,0.650,0.267,0.571


## Cell 5: Perkiraan "hati-hati" dengan regresi quantile

**Ide:** di dapur bakery, kehabisan bahan lebih merugikan daripada sisa sedikit lebih banyak. Model biasa memperkirakan nilai rata-rata. Regresi **quantile 0,7** memperkirakan nilai yang **70% kemungkinan tidak terlampaui**, sehingga perkiraannya sedikit lebih tinggi dari rata-rata dan lebih sering memicu Restock tepat waktu.

**Pembanding yang adil:** baseline juga digeser naik (× 1,05 dan × 1,10). Tujuannya membuktikan bahwa keunggulan model bukan sekadar karena "menebak lebih tinggi".

**Hasil di validasi:**
- Di antara model GB, quantile 0,7 memberi recall Restock tertinggi (0,697). Quantile 0,6 memberi F1 macro tertinggi (0,616).
- Baseline yang digeser naik meningkatkan recall Restock, tetapi **tetap tidak pernah mendeteksi Excess** (F1 Excess 0).

**Keputusan:** GB quantile 0,7 dipilih karena prioritas proyek adalah **tidak melewatkan kondisi Restock**, sambil tetap mampu mendeteksi Excess. Keputusan ini dikunci sebelum data test dibuka.

In [ ]:
# CELL 5 : Eksperimen tebakan hati-hati (quantile) vs baseline yang digeser

hasil5 = {}

# 1. Gradient Boosting yang menebak quantile tertentu (bukan rata-rata)
for q in [0.5, 0.6, 0.7]:
    m = HistGradientBoostingRegressor(loss="quantile", quantile=q, random_state=SEED)
    m.fit(X_tr, y_tr)
    hasil5[f"GB quantile {q}"] = evaluasi(va, m.predict(X_va))

# 2. Pembanding adil: baseline yang tebakannya juga digeser naik
for k in [1.00, 1.05, 1.10]:
    hasil5[f"Baseline x {k:.2f}"] = evaluasi(va, np.full(len(va), H * k))

print("==== Perbandingan di data validasi ====")
display(pd.DataFrame(hasil5).T.round(3))

# 3. Konteks: berapa banyak tiap kelas di data validasi
print("\nJumlah kelas di validasi:")
print(va["label"].value_counts())

==== Perbandingan di data validasi ====


,meleset_hari,recall_restock,f1_restock,f1_excess,f1_macro
GB quantile 0.5,1.825,0.559,0.648,0.286,0.582
GB quantile 0.6,1.835,0.631,0.677,0.364,0.616
GB quantile 0.7,1.829,0.697,0.706,0.182,0.565
Baseline x 1.00,1.719,0.661,0.728,0.000,0.524
Baseline x 1.05,1.718,0.706,0.732,0.000,0.521
Baseline x 1.10,1.774,0.742,0.731,0.000,0.516



Jumlah kelas di validasi:
label
Sufficient    723
Restock       469
Excess          8
Name: count, dtype: int64


## Cell 6: Uji hipotesis, apakah fitur stok mengganggu?

**Hipotesis:** fitur stok (stok gudang, sedang dikirim, posisi stok) mungkin tidak berhubungan dengan pemakaian dan malah mengganggu model.

**Cara menguji:** model dengan semua fitur dibandingkan dengan model yang hanya memakai fitur permintaan (`tren_7_vs_30` dan `bulan`).

**Hasil:** model dengan fitur permintaan saja justru **lebih buruk** (meleset lebih besar, recall Restock lebih rendah). Hipotesis ditolak, jadi **semua fitur tetap dipakai**.

In [ ]:
# CELL 6 : Uji hipotesis: apakah fitur stok justru mengganggu model pemakaian?

FITUR_PERMINTAAN = ["tren_7_vs_30", "bulan"]

hasil6 = {}

# 1. Bandingkan model dengan semua fitur vs hanya fitur permintaan
for nama_set, kolom in [("semua fitur", FITUR_MODEL), ("fitur permintaan", FITUR_PERMINTAAN)]:
    for q in [0.6, 0.7]:
        m = HistGradientBoostingRegressor(loss="quantile", quantile=q, random_state=SEED)
        m.fit(X_tr[kolom], y_tr)
        hasil6[f"GB q{q} | {nama_set}"] = evaluasi(va, m.predict(X_va[kolom]))

# 2. Pembanding: baseline yang digeser
for k in [1.05, 1.10]:
    hasil6[f"Baseline x {k:.2f}"] = evaluasi(va, np.full(len(va), H * k))

# 3. Tampilkan hanya metrik yang bisa dipercaya di validasi (Excess cuma 8 baris)
print("==== Perbandingan di validasi (fokus Restock) ====")
display(pd.DataFrame(hasil6).T[["meleset_hari", "recall_restock", "f1_restock"]].round(3))

==== Perbandingan di validasi (fokus Restock) ====


,meleset_hari,recall_restock,f1_restock
GB q0.6 | semua fitur,1.835,0.631,0.677
GB q0.7 | semua fitur,1.829,0.697,0.706
GB q0.6 | fitur permintaan,1.909,0.612,0.652
GB q0.7 | fitur permintaan,1.985,0.672,0.680
Baseline x 1.05,1.718,0.706,0.732
Baseline x 1.10,1.774,0.742,0.731


## Cell 7: Evaluasi akhir di data test (dibuka satu kali)

**Langkah:**
1. Tiga kandidat dikunci: baseline, baseline + margin 5%, dan GB quantile 0,7.
2. Model final dilatih ulang di **seluruh data train** (train dalam + validasi), karena pilihannya sudah dikunci.
3. Semua kandidat dinilai di data test.

**Hasil di test (1.160 baris):**

| Metrik | Baseline | Baseline + 5% | **GB quantile 0,7** |
|---|---|---|---|
| Rata-rata meleset (hari) | 2,65 | 2,69 | **1,31** |
| Recall Restock | 0,516 | 0,574 | **0,897** |
| F1 Restock | 0,562 | 0,588 | **0,860** |
| F1 Excess | 0,000 | 0,000 | **0,731** |
| F1 macro | 0,434 | 0,441 | **0,828** |

**Cara membaca:**
- Dari 380 kondisi Restock di test, model mendeteksi sekitar 90%, sedangkan baseline hanya sekitar separuhnya.
- Baseline + 5% hanya sedikit lebih baik dari baseline. Keunggulan model **bukan** karena sekadar menebak lebih tinggi.
- Excess mulai terdeteksi (precision 0,964, recall 0,589). Model jarang salah menyebut Excess, tetapi masih melewatkan sebagian.

**Catatan penting:** hasil test (F1 macro 0,828) jauh lebih tinggi dari hasil validasi (0,565). Perbedaan ini diselidiki di Cell 8.

In [ ]:
# CELL 7 : Kunci kandidat, latih ulang di seluruh train, evaluasi SEKALI di test

# 1. Kandidat yang sudah dikunci dari hasil validasi (Cell 4 sampai 6)
KANDIDAT = {
    "Baseline (rata-rata x 7)": ("baseline", 1.00),
    "Baseline + margin 5%":     ("baseline", 1.05),
    "GB quantile 0.7":          ("model", 0.7),
}

# 2. Latih ulang model final di SELURUH train (train dalam + validasi)
#    Pilihan sudah dikunci, jadi validasi boleh ikut dipakai untuk belajar
model_final = HistGradientBoostingRegressor(loss="quantile", quantile=0.7, random_state=SEED)
model_final.fit(X_train, y_train)

# 3. Evaluasi semua kandidat di test
hasil_test = {}
label_test = {}
for nama, (jenis, param) in KANDIDAT.items():
    if jenis == "baseline":
        pred = np.full(len(test), H * param)
    else:
        pred = model_final.predict(X_test)
    hasil_test[nama] = evaluasi(test, pred)
    _, label_test[nama] = policy(test, pred * test["avg_usage_30"])

print("==== Ringkasan di data test ====")
display(pd.DataFrame(hasil_test).T.round(3))

# 4. Laporan lengkap per kelas untuk tiap kandidat
for nama in KANDIDAT:
    print(f"\n==== {nama} ====")
    print(classification_report(test["label"], label_test[nama], labels=LABELS, digits=3, zero_division=0))

==== Ringkasan di data test ====


,meleset_hari,recall_restock,f1_restock,f1_excess,f1_macro
Baseline (rata-rata x 7),2.652,0.516,0.562,0.000,0.434
Baseline + margin 5%,2.687,0.574,0.588,0.000,0.441
GB quantile 0.7,1.306,0.897,0.860,0.731,0.828



==== Baseline (rata-rata x 7) ====
              precision    recall  f1-score   support

     Restock      0.616     0.516     0.562       380
  Sufficient      0.673     0.819     0.739       690
      Excess      0.000     0.000     0.000        90

    accuracy                          0.656      1160
   macro avg      0.430     0.445     0.434      1160
weighted avg      0.602     0.656     0.624      1160


==== Baseline + margin 5% ====
              precision    recall  f1-score   support

     Restock      0.604     0.574     0.588       380
  Sufficient      0.685     0.793     0.735       690
      Excess      0.000     0.000     0.000        90

    accuracy                          0.659      1160
   macro avg      0.429     0.455     0.441      1160
weighted avg      0.605     0.659     0.630      1160


==== GB quantile 0.7 ====
              precision    recall  f1-score   support

     Restock      0.826     0.897     0.860       380
  Sufficient      0.890     0.893 

## Cell 8: Diagnosis, kenapa hasil test lebih baik dari validasi?

Cell ini **hanya untuk memahami hasil**. Model tidak diganti setelah melihat data test.

**Temuan:**
1. **Fitur terpenting** adalah `tren_7_vs_30` dan `bulan`. Model terutama belajar membaca perubahan tren dan pola musim.
2. **Bukan karena tambahan data training.** Model yang hanya belajar sampai Januari 2022 memberi F1 macro test yang sama (0,828).
3. **Periode test lebih bergejolak.** Meleset baseline per bulan: Juni 1,13, **Juli 4,06**, Agustus 2,38, **September 3,08** hari. Di bulan-bulan yang bergejolak, baseline (rata-rata 30 hari) tertinggal, sedangkan model membaca tren terbaru. Periode validasi (Februari sampai Mei) lebih stabil, sehingga baseline sudah cukup baik di sana.
4. Model juga lebih unggul saat stok gudang sangat tipis (kurang dari 1 hari).

**Kesimpulan diagnosis:** selisih validasi dan test berasal dari **karakter periode**, bukan dari kebocoran data. Konsekuensinya, angka 0,828 bergantung pada periode Juni sampai September 2022 dan bisa berbeda di periode lain.

### Checklist audit data leakage

| Potensi kebocoran | Pengecekan |
|---|---|
| Fitur memakai informasi hari yang sama atau masa depan | semua fitur histori dibuat dengan `shift(1)` di Notebook 1 |
| Ambang bulk order dan safety stock dihitung dari seluruh data | dihitung **hanya dari data train** di Notebook 1 |
| Pembagian data acak (masa depan masuk ke train) | pembagian **berdasarkan tanggal** (`SPLIT_DATE`, `VAL_DATE`) |
| Model dipilih setelah melihat test | kandidat dikunci di Cell 7 sebelum test dibuka, dan test dibuka sekali |
| Model "menghafal" periode test | model yang belajar sampai Januari 2022 memberi hasil test yang sama |

In [ ]:
# CELL 8 : Diagnosis, kenapa model jauh lebih baik di test daripada di validasi?
# (Hanya untuk memahami hasil. Model TIDAK diganti.)

from sklearn.inspection import permutation_importance

# 1. Fitur mana yang paling dipakai model di data test
imp = permutation_importance(
    model_final, X_test, y_test,
    n_repeats=10, random_state=SEED, scoring="neg_mean_absolute_error",
)
print("==== Pentingnya fitur (makin besar makin penting) ====")
display(pd.Series(imp.importances_mean, index=FITUR_MODEL).sort_values(ascending=False).round(3))

# 2. Uji dugaan 1: model yang hanya belajar sampai Januari 2022, dinilai di test
m_dalam = HistGradientBoostingRegressor(loss="quantile", quantile=0.7, random_state=SEED)
m_dalam.fit(X_tr, y_tr)

banding = {
    "GB final (belajar s/d Mei 2022) di test":   evaluasi(test, model_final.predict(X_test)),
    "GB dalam (belajar s/d Jan 2022) di test":   evaluasi(test, m_dalam.predict(X_test)),
    "GB dalam (belajar s/d Jan 2022) di validasi": evaluasi(va, m_dalam.predict(X_va)),
}
print("\n==== Pengaruh periode belajar ====")
display(pd.DataFrame(banding).T.round(3))

# 3. Meleset per bulan di test: baseline vs model
cek = test[["date", "target_rasio"]].copy()
cek["err_baseline"] = (H - cek["target_rasio"]).abs()
cek["err_model"]    = (model_final.predict(X_test) - cek["target_rasio"]).abs()
print("\n==== Meleset rata-rata per bulan di test (hari) ====")
display(cek.groupby(cek["date"].dt.to_period("M"))[["err_baseline", "err_model"]].mean().round(2))

# 4. Uji dugaan 2: apakah model unggul terutama saat stok gudang sangat tipis?
cek["stok_tipis"] = X_test["stok_cover_now"] < 1
print("\n==== Meleset saat stok gudang tipis (< 1 hari) vs normal ====")
display(cek.groupby("stok_tipis")[["err_baseline", "err_model"]].agg(["mean", "count"]).round(2))

==== Pentingnya fitur (makin besar makin penting) ====


,0
tren_7_vs_30,1.335
bulan,0.549
stok_cover_now,0.040
posisi_stok,0.037
days_since_last_order,0.010
on_order_hari,0.008



==== Pengaruh periode belajar ====


,meleset_hari,recall_restock,f1_restock,f1_excess,f1_macro
GB final (belajar s/d Mei 2022) di test,1.306,0.897,0.860,0.731,0.828
GB dalam (belajar s/d Jan 2022) di test,1.291,0.884,0.846,0.753,0.828
GB dalam (belajar s/d Jan 2022) di validasi,1.829,0.697,0.706,0.182,0.565



==== Meleset rata-rata per bulan di test (hari) ====


,err_baseline,err_model
date,,
2022-06,1.13,1.07
2022-07,4.06,1.89
2022-08,2.38,1.46
2022-09,3.08,0.66



==== Meleset saat stok gudang tipis (< 1 hari) vs normal ====


err_baseline       err_model      
                   mean count      mean count
stok_tipis                                   
False              2.58  1102      1.28  1102
True               4.06    58      1.88    58

## Cell 9: Simpan model final beserta "kartu identitasnya"

Model disimpan bersama informasi pendukung dalam satu file `joblib`:
- daftar dan urutan fitur yang harus dipakai,
- definisi target dan nilai quantile,
- seluruh parameter simulasi (`config`),
- metrik test dari Cell 7,
- versi scikit-learn (1.6.1).

**Kenapa perlu?** Notebook 3 (GenAI) memuat file ini lalu **membuktikan ulang** bahwa F1 macro yang dihasilkan sama persis (0,828). Dengan begitu, model yang dipakai untuk laporan GenAI dijamin sama dengan model yang dievaluasi di sini.

In [ ]:
# CELL 9 : Simpan model final beserta "kartu identitas"-nya

import joblib
import sklearn
import datetime

bundle = {
    "model": model_final,
    "fitur_model": FITUR_MODEL,
    "target": "target_rasio = pemakaian 7 hari ke depan / avg_usage_30",
    "quantile": 0.7,
    "config": CFG,                       # parameter simulasi dan aturan label
    "metrik_test": hasil_test,           # hasil Cell 7, untuk dokumentasi
    "sklearn_version": sklearn.__version__,
    "dibuat": datetime.datetime.now().isoformat(timespec="seconds"),
}

joblib.dump(bundle, MODEL_DIR / "usage_model_gb_q07.joblib")
pd.DataFrame(hasil_test).T.round(3).to_csv(MODEL_DIR / "hasil_evaluasi_test.csv")

print("Tersimpan:", MODEL_DIR / "usage_model_gb_q07.joblib")
print("Tersimpan:", MODEL_DIR / "hasil_evaluasi_test.csv")
print("Versi scikit-learn:", sklearn.__version__)

Tersimpan: /content/drive/MyDrive/Tewiwi-Bakehouse/models/usage_model_gb_q07.joblib
Tersimpan: /content/drive/MyDrive/Tewiwi-Bakehouse/models/hasil_evaluasi_test.csv
Versi scikit-learn: 1.6.1


## Kesimpulan RQ2

1. Model **HistGradientBoostingRegressor dengan quantile 0,7** yang dimasukkan ke inventory policy menghasilkan status stok yang jauh lebih tepat dibanding baseline: F1 macro **0,828 vs 0,434**, recall Restock **0,897 vs 0,516**.
2. Model mampu mendeteksi kondisi **Excess** (F1 0,731), sesuatu yang tidak bisa dilakukan baseline.
3. Keunggulan model bukan karena sekadar menebak lebih tinggi, karena baseline yang digeser naik 5% hanya naik tipis.

## Keterbatasan
- **Data simulasi:** pemakaian bahan disimulasikan dari data penjualan bakery publik (Kaggle) dan resep yang dipetakan ke produk Tewiwi. Hasil di data asli Tewiwi bisa berbeda.
- **Label dibuat oleh policy:** label adalah hasil aturan inventory, bukan keputusan nyata owner. Kualitas label bergantung pada asumsi parameter (lead time, safety stock, waste).
- **Hasil bergantung periode:** selisih besar antara validasi dan test menunjukkan performa dipengaruhi karakter periode. Evaluasi dengan beberapa periode (rolling validation) akan memberi gambaran yang lebih kuat.
- **Kelas Excess sedikit:** 8 baris di validasi dan 90 baris di test, sehingga metrik Excess masih kurang stabil.

## Rencana pengembangan
- Validasi bergulir di beberapa periode waktu.
- Membandingkan dengan model deret waktu (misalnya ARIMA) sebagai pembanding tambahan.
- Melatih ulang dengan data transaksi asli Tewiwi bila tersedia.